# Notebook 15 - CSD100 Three-Variant PFB + SAC Experiment

This notebook is intentionally thin. All experiment logic lives in `src/var_soict/*.py`; the notebook only calls those functions/classes and displays outputs.

Variants:

```text
1. Original PFB + SAC: F3 / zero-based scale 2, full SVD component
2. Global Top-2 PFB + SAC: scales [0,1,2,9]
3. Global Background + Object-Masked Foreground PFB + SAC: global [0,1,2], masked [3,6,9]
```


## 1. Import Project Modules And Build Configuration

In [ ]:
from pathlib import Path
import sys

VAR_SOICT_ROOT = Path('/content/VAR_SOICT') if Path('/content/VAR_SOICT').exists() else Path.cwd().resolve()
if not (VAR_SOICT_ROOT / 'src' / 'var_soict').exists():
    VAR_SOICT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src' / 'var_soict').exists())
sys.path.insert(0, str(VAR_SOICT_ROOT / 'src'))

from var_soict.config import ExperimentConfig
from var_soict.bootstrap import build_runtime_paths

config = ExperimentConfig(
    root=Path('/content/notebook_15_pfb_sac_3variants_csd100'),
    infinity_source_dir=VAR_SOICT_ROOT / 'Infinity',
    download_missing_model_files=True,
    output_run_name='notebook_15_pfb_sac_3variants_csd100_20pairs_seed2026',
    model_pn='0.25M',
    cfg=1.0,
    tau=0.1,
    top_k=600,
    top_p=0.95,
    seed=2026,
    t5_device='cuda',
    object_mask_device='cuda',
    style_reference_mask_prompt='main object',
)
paths = build_runtime_paths(config)

print('VAR_SOICT root:', VAR_SOICT_ROOT)
print('Runtime root:', paths.runtime_root)
print('Output dir:', paths.output_dir)
print('Official Infinity source:', paths.official_dir)
print('GGUF runtime cache:', paths.root)
print('Seed:', config.seed)
print('Model preset:', config.model_pn)


## 2. Runtime And Dependencies

In [ ]:
from var_soict.bootstrap import check_torch_runtime, install_dependencies

DEVICE = check_torch_runtime(require_cuda=True)
install_dependencies()


## 3. Download/Patch Infinity-2B GGUF Assets

In [ ]:
from var_soict.bootstrap import import_gguf_loader, prepare_infinity_sources, validate_infinity_runtime_imports, verify_model_files

model_files = prepare_infinity_sources(config, paths)
verify_model_files(paths, model_files)
gguf_loader = import_gguf_loader(paths, model_files)
validate_infinity_runtime_imports(paths)


## 4. Load Model Components And Scale Schedule

In [ ]:
from var_soict.bootstrap import build_scale_schedule, load_model_bundle

bundle = load_model_bundle(config, model_files, gguf_loader)
bundle.scale_schedule = build_scale_schedule(config.model_pn, aspect_ratio=1.0)
print('Scale schedule length:', len(bundle.scale_schedule))
print('Scale schedule:', bundle.scale_schedule)


## 5. Create CSD100 Notebook 15 Experiment

In [ ]:
import pandas as pd

from var_soict.notebook15_csd100 import build_notebook15_experiment
from var_soict.style_transfer import StyleTransferEngine

engine = StyleTransferEngine(bundle, config)
experiment = build_notebook15_experiment(
    engine=engine,
    config=config,
    paths=paths,
    var_soict_root=VAR_SOICT_ROOT,
)

pair_table = pd.DataFrame([{key: str(value) for key, value in row.items()} for row in experiment.pair_rows])
display(pair_table[['pair_id', 'content_id', 'content_object', 'content_style', 'style_id', 'style_object', 'style_label', 'pfb_prompt']])


## 6. Generate Baselines And CLIPSeg Masks

In [ ]:
baseline_results, content_masks, style_masks = experiment.run_baselines_and_masks(force=False)


## 7. Run Variant 1 - Original PFB + SAC

In [ ]:
original_results = experiment.run_original_pfb_sac(force=False)


## 8. Run Variant 2 - Global Top-2 PFB + SAC

In [ ]:
global_top2_results = experiment.run_global_top2(force=False)


## 9. Run Variant 3 - Global Background + Object-Masked Foreground PFB + SAC

In [ ]:
object_masked_results = experiment.run_object_masked(force=False)


## 10. Scale Trajectory Diagnostics

In [ ]:
experiment.plot_all_scale_diagnostics()


## 11. Final Gallery Comparison

In [ ]:
gallery_path = experiment.show_final_gallery()
print('Gallery:', gallery_path)


## 12. Style Metrics: `S_txt`, `S_img`, `S_harmonic`

In [ ]:
from var_soict.clip_metrics import CLIPMetricsEvaluator

metrics = CLIPMetricsEvaluator(output_dir=experiment.dirs.evaluation)
style_metrics_df = metrics.compute_style_metrics(experiment)
style_metrics_df, style_summary_df = metrics.save_style_metrics(style_metrics_df)


## 13. Content Preservation And Style-Object Leakage Metrics

In [ ]:
content_leakage_df = metrics.compute_content_leakage_metrics(experiment)
content_leakage_df, content_leakage_summary_df = metrics.save_content_leakage_metrics(content_leakage_df)


## 14. Output Locations And Download Package

In [ ]:
print('Output dir:', experiment.dirs.root)
print('Final gallery:', experiment.dirs.galleries)
print('Diagnostics:', experiment.dirs.diagnostics)
print('Evaluation:', experiment.dirs.evaluation)
zip_path = experiment.package_outputs(include_traces=False)
print('ZIP:', zip_path)
